In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
import torch
from transformers import SamProcessor, SamModel
import xml.dom.minidom as minidom

# ==== CẤU HÌNH ====
input_image_dir = "VESSELimg/Pilot1_split/part1"
input_xml_dir = "VESSELimg/Pilot1_split/part1"
#background_dir = "/home/aiplatform/projects/test/data/UAV_SEA_OUTPUT"
background_dir = "BackGround"
output_dir = "SAM_transformed_outputs"
os.makedirs(output_dir, exist_ok=True)

# ==== BIẾN ĐỔI ====
transformations = [
    {"name": "rotate_30", "angle": 30, "scale": 1.0, "offset": (0, 0)},
    {"name": "rotate_-15", "angle": -15, "scale": 1.0, "offset": (0, 0)},
    {"name": "scale_up", "angle": 0, "scale": 1.3, "offset": (0, 0)},
    {"name": "scale_down", "angle": 0, "scale": 0.7, "offset": (0, 0)},
    {"name": "shift_right", "angle": 0, "scale": 1.0, "offset": (50, 0)},
    {"name": "shift_down", "angle": 0, "scale": 1.0, "offset": (0, 50)},
]

# ==== HÀM HỖ TRỢ ====
def read_pilotbboxes_from_voc(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    bboxes = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name.lower() == 'pilot':
            bbox = obj.find("bndbox")
            x_min = int(bbox.find("xmin").text)
            y_min = int(bbox.find("ymin").text)
            x_max = int(bbox.find("xmax").text)
            y_max = int(bbox.find("ymax").text)
            bboxes.append({"label": name, "box": [x_min, y_min, x_max, y_max]})
    return bboxes

def create_voc_xml(filename, path, width, height, depth, label, bbox, save_path):
    doc = minidom.Document()
    annotation = doc.createElement("annotation")
    doc.appendChild(annotation)

    def append_text_node(name, text, parent):
        node = doc.createElement(name)
        node.appendChild(doc.createTextNode(str(text)))
        parent.appendChild(node)

    append_text_node("folder", "", annotation)
    append_text_node("filename", filename, annotation)
    append_text_node("path", path, annotation)

    source = doc.createElement("source")
    append_text_node("database", "roboflow.com", source)
    annotation.appendChild(source)

    size = doc.createElement("size")
    append_text_node("width", width, size)
    append_text_node("height", height, size)
    append_text_node("depth", depth, size)
    annotation.appendChild(size)

    append_text_node("segmented", 0, annotation)

    obj = doc.createElement("object")
    append_text_node("name", label, obj)
    append_text_node("pose", "Unspecified", obj)
    append_text_node("truncated", 0, obj)
    append_text_node("difficult", 0, obj)
    append_text_node("occluded", 0, obj)

    bbox_tag = doc.createElement("bndbox")
    xmin, ymin, xmax, ymax = bbox
    append_text_node("xmin", xmin, bbox_tag)
    append_text_node("xmax", xmax, bbox_tag)
    append_text_node("ymin", ymin, bbox_tag)
    append_text_node("ymax", ymax, bbox_tag)
    obj.appendChild(bbox_tag)
    annotation.appendChild(obj)

    with open(save_path, "w") as f:
        f.write(doc.toprettyxml(indent="  "))

# ==== LOAD SAM ====
processor = SamProcessor.from_pretrained("facebook/sam-vit-huge")
model = SamModel.from_pretrained("facebook/sam-vit-huge").cuda()

# ==== DUYỆT ẢNH ====
for image_file in os.listdir(input_image_dir):
    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(input_image_dir, image_file)
    xml_path = os.path.join(input_xml_dir, os.path.splitext(image_file)[0] + ".xml")

    if not os.path.exists(xml_path):
        print(f"❌ Không tìm thấy XML cho: {image_file}")
        continue

    image = Image.open(image_path).convert("RGB")
    bboxes = read_pilotbboxes_from_voc(xml_path)
    if not bboxes:
        continue

    for i, item in enumerate(bboxes):
        box = [[list(map(float, item["box"]))]]
        inputs = processor(image, input_boxes=box, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model(**inputs)
            masks = processor.image_processor.post_process_masks(
                outputs.pred_masks.cpu(),
                inputs["original_sizes"].cpu(),
                inputs["reshaped_input_sizes"].cpu()
            )
            mask_tensor = masks[0].squeeze(0)
            mask = mask_tensor[0].numpy().astype(np.uint8)

        masked_obj = np.array(image) * mask[:, :, None]
        alpha = (mask * 255).astype(np.uint8)
        obj_rgba = np.dstack((masked_obj, alpha))

        x1, y1, x2, y2 = item["box"]
        obj_crop = obj_rgba[y1:y2, x1:x2]
        obj_pil = Image.fromarray(obj_crop)

        for bg_name in os.listdir(background_dir):
            if not bg_name.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            bg_path = os.path.join(background_dir, bg_name)
            background = Image.open(bg_path).convert("RGBA").resize(image.size)

            for t in transformations:
                trans_name = t["name"]
                angle = t["angle"]
                scale = t["scale"]
                offset_x, offset_y = t["offset"]

                ow, oh = obj_pil.size
                new_size = (int(ow * scale), int(oh * scale))
                obj_scaled = obj_pil.resize(new_size, resample=Image.BILINEAR)
                obj_transformed = obj_scaled.rotate(angle, expand=True)

                bg_copy = background.copy()
                bg_w, bg_h = bg_copy.size
                obj_w, obj_h = obj_transformed.size
                cx = (bg_w - obj_w) // 2 + offset_x
                cy = (bg_h - obj_h) // 2 + offset_y
                bg_copy.paste(obj_transformed, (cx, cy), obj_transformed)

                base_name = os.path.splitext(image_file)[0]
                bg_base = os.path.splitext(bg_name)[0]
                out_name = f"{base_name}_{i}_{item['label']}_{bg_base}_{trans_name}.png"
                out_path = os.path.join(output_dir, out_name)
                bg_copy.save(out_path)
                print(f"✅ Lưu: {out_path}")

                xml_out_path = os.path.join(output_dir, os.path.splitext(out_name)[0] + ".xml")
                create_voc_xml(
                    filename=out_name,
                    path=out_path,
                    width=bg_w,
                    height=bg_h,
                    depth=3,
                    label=item['label'],
                    bbox=[cx, cy, cx + obj_w, cy + obj_h],
                    save_path=xml_out_path
                )
                print(f"📝 Lưu annotation: {xml_out_path}")


In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
import torch
from transformers import SamProcessor, SamModel
import xml.dom.minidom as minidom

# ==== CẤU HÌNH ====
input_image_dir = "/home/aiplatform/projects/test/data/VESSELimg/Pilot1_split/part4"
input_xml_dir = "/home/aiplatform/projects/test/data/VESSELimg/Pilot1_split/part4"
background_dir = "BackGround"
output_dir = "test_combined_outputs/part2"
os.makedirs(output_dir, exist_ok=True)

# ==== HÀM HỖ TRỢ ====
def get_pilot_and_largest(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    bboxes = []
    largest_bbox = None
    largest_area = 0

    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        bndbox = obj.find("bndbox")
        xmin = int(bndbox.find("xmin").text)
        xmax = int(bndbox.find("xmax").text)
        ymin = int(bndbox.find("ymin").text)
        ymax = int(bndbox.find("ymax").text)
        area = (xmax - xmin) * (ymax - ymin)

        bbox = {"label": name, "box": [xmin, ymin, xmax, ymax]}
        if name.lower() == "pilot":
            bboxes.append(bbox)
        if area > largest_area:
            largest_area = area
            largest_bbox = bbox

    if largest_bbox and largest_bbox not in bboxes:
        bboxes.append(largest_bbox)

    return bboxes

def create_voc_xml(filename, path, width, height, depth, labels_boxes, save_path):
    doc = minidom.Document()
    annotation = doc.createElement("annotation")
    doc.appendChild(annotation)

    def append_text_node(name, text, parent):
        node = doc.createElement(name)
        node.appendChild(doc.createTextNode(str(text)))
        parent.appendChild(node)

    append_text_node("folder", "", annotation)
    append_text_node("filename", filename, annotation)
    append_text_node("path", path, annotation)

    source = doc.createElement("source")
    append_text_node("database", "roboflow.com", source)
    annotation.appendChild(source)

    size = doc.createElement("size")
    append_text_node("width", width, size)
    append_text_node("height", height, size)
    append_text_node("depth", depth, size)
    annotation.appendChild(size)

    append_text_node("segmented", 0, annotation)

    for label, bbox in labels_boxes:
        obj = doc.createElement("object")
        append_text_node("name", label, obj)
        append_text_node("pose", "Unspecified", obj)
        append_text_node("truncated", 0, obj)
        append_text_node("difficult", 0, obj)
        append_text_node("occluded", 0, obj)

        bbox_tag = doc.createElement("bndbox")
        xmin, ymin, xmax, ymax = bbox
        append_text_node("xmin", xmin, bbox_tag)
        append_text_node("xmax", xmax, bbox_tag)
        append_text_node("ymin", ymin, bbox_tag)
        append_text_node("ymax", ymax, bbox_tag)
        obj.appendChild(bbox_tag)
        annotation.appendChild(obj)

    with open(save_path, "w") as f:
        f.write(doc.toprettyxml(indent="  "))

# ==== LOAD SAM ====
processor = SamProcessor.from_pretrained("facebook/sam-vit-huge")
model = SamModel.from_pretrained("facebook/sam-vit-huge").cuda()

# ==== DUYỆT ẢNH ====
for image_file in os.listdir(input_image_dir):
    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(input_image_dir, image_file)
    xml_path = os.path.join(input_xml_dir, os.path.splitext(image_file)[0] + ".xml")

    if not os.path.exists(xml_path):
        print(f"❌ Không tìm thấy XML cho: {image_file}")
        continue

    image = Image.open(image_path).convert("RGB")
    bboxes = get_pilot_and_largest(xml_path)
    if not bboxes:
        continue

    # === Chọn ảnh nền bất kỳ và không resize ===
    bg_files = [f for f in os.listdir(background_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if not bg_files:
        print("❌ Không có ảnh nền.")
        continue
    background = Image.open(os.path.join(background_dir, bg_files[0])).convert("RGBA")
    bg_copy = background.copy()

    # === Tính tỉ lệ scale giữa ảnh gốc và ảnh nền ===
    scale_x = bg_copy.width / image.width
    scale_y = bg_copy.height / image.height

    annotations = []

    for item in bboxes:
        box = [[list(map(float, item["box"]))]]
        inputs = processor(image, input_boxes=box, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model(**inputs)
            masks = processor.image_processor.post_process_masks(
                outputs.pred_masks.cpu(),
                inputs["original_sizes"].cpu(),
                inputs["reshaped_input_sizes"].cpu()
            )
            mask_tensor = masks[0].squeeze(0)
            mask = mask_tensor[0].numpy().astype(np.uint8)

        masked_obj = np.array(image) * mask[:, :, None]
        alpha = (mask * 255).astype(np.uint8)
        obj_rgba = np.dstack((masked_obj, alpha))

        x1, y1, x2, y2 = item["box"]
        obj_crop = obj_rgba[y1:y2, x1:x2]
        obj_pil = Image.fromarray(obj_crop)

        # === Scale vị trí và kích thước ===
        scaled_x1 = int(x1 * scale_x)
        scaled_y1 = int(y1 * scale_y)
        scaled_x2 = int(x2 * scale_x)
        scaled_y2 = int(y2 * scale_y)

        new_w = scaled_x2 - scaled_x1
        new_h = scaled_y2 - scaled_y1
        obj_pil = obj_pil.resize((new_w, new_h), Image.LANCZOS)

        # === Dán vào ảnh nền tại vị trí mới ===
        bg_copy.paste(obj_pil, (scaled_x1, scaled_y1), obj_pil)
        annotations.append((item["label"], [scaled_x1, scaled_y1, scaled_x2, scaled_y2]))

    # === Lưu ảnh và XML ===
    base_name = os.path.splitext(image_file)[0]
    out_image_name = f"{base_name}_combined.png"
    out_image_path = os.path.join(output_dir, out_image_name)
    bg_copy.save(out_image_path)

    xml_out_path = os.path.join(output_dir, f"{base_name}_combined.xml")
    create_voc_xml(
        filename=out_image_name,
        path=out_image_path,
        width=bg_copy.width,
        height=bg_copy.height,
        depth=3,
        labels_boxes=annotations,
        save_path=xml_out_path
    )

    print(f"✅ Đã lưu: {out_image_path}")
    print(f"📝 Annotation: {xml_out_path}")


In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
import torch
from transformers import SamProcessor, SamModel
import xml.dom.minidom as minidom

# ==== CẤU HÌNH ====

input_image_dir = "/home/aiplatform/projects/test/data/VESSELimg/Pilot1_split/part5"
input_xml_dir = "/home/aiplatform/projects/test/data/VESSELimg/Pilot1_split/part5"
background_dir = "BackGround"
output_dir = "test_combined_outputs/part5"
os.makedirs(output_dir, exist_ok=True)

# ==== HÀM HỖ TRỢ ====
def get_pilot_and_largest(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    bboxes = []
    largest_bbox = None
    largest_area = 0

    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        bndbox = obj.find("bndbox")
        xmin = int(bndbox.find("xmin").text)
        xmax = int(bndbox.find("xmax").text)
        ymin = int(bndbox.find("ymin").text)
        ymax = int(bndbox.find("ymax").text)
        area = (xmax - xmin) * (ymax - ymin)

        bbox = {"label": name, "box": [xmin, ymin, xmax, ymax]}
        if name.lower() == "pilot":
            bboxes.append(bbox)
        if area > largest_area:
            largest_area = area
            largest_bbox = bbox

    if largest_bbox and largest_bbox not in bboxes:
        bboxes.append(largest_bbox)

    return bboxes

def create_voc_xml(filename, path, width, height, depth, labels_boxes, save_path):
    doc = minidom.Document()
    annotation = doc.createElement("annotation")
    doc.appendChild(annotation)

    def append_text_node(name, text, parent):
        node = doc.createElement(name)
        node.appendChild(doc.createTextNode(str(text)))
        parent.appendChild(node)

    append_text_node("folder", "", annotation)
    append_text_node("filename", filename, annotation)
    append_text_node("path", path, annotation)

    source = doc.createElement("source")
    append_text_node("database", "roboflow.com", source)
    annotation.appendChild(source)

    size = doc.createElement("size")
    append_text_node("width", width, size)
    append_text_node("height", height, size)
    append_text_node("depth", depth, size)
    annotation.appendChild(size)

    append_text_node("segmented", 0, annotation)

    for label, bbox in labels_boxes:
        obj = doc.createElement("object")
        append_text_node("name", label, obj)
        append_text_node("pose", "Unspecified", obj)
        append_text_node("truncated", 0, obj)
        append_text_node("difficult", 0, obj)
        append_text_node("occluded", 0, obj)

        bbox_tag = doc.createElement("bndbox")
        xmin, ymin, xmax, ymax = bbox
        append_text_node("xmin", xmin, bbox_tag)
        append_text_node("xmax", xmax, bbox_tag)
        append_text_node("ymin", ymin, bbox_tag)
        append_text_node("ymax", ymax, bbox_tag)
        obj.appendChild(bbox_tag)
        annotation.appendChild(obj)

    with open(save_path, "w") as f:
        f.write(doc.toprettyxml(indent="  "))

# ==== LOAD SAM ====
processor = SamProcessor.from_pretrained("facebook/sam-vit-huge")
model = SamModel.from_pretrained("facebook/sam-vit-huge").cuda()

# ==== DUYỆT ẢNH ====
for image_file in os.listdir(input_image_dir):
    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(input_image_dir, image_file)
    xml_path = os.path.join(input_xml_dir, os.path.splitext(image_file)[0] + ".xml")

    if not os.path.exists(xml_path):
        print(f"❌ Không tìm thấy XML cho: {image_file}")
        continue

    image = Image.open(image_path).convert("RGB")
    bboxes = get_pilot_and_largest(xml_path)
    if not bboxes:
        continue

    for bg_file in os.listdir(background_dir):
        if not bg_file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        background = Image.open(os.path.join(background_dir, bg_file)).convert("RGBA")
        bg_copy = background.copy()

        # === Tính tỉ lệ scale giữa ảnh gốc và ảnh nền ===
        scale_x = bg_copy.width / image.width
        scale_y = bg_copy.height / image.height

        annotations = []

        for item in bboxes:
            box = [[list(map(float, item["box"]))]]
            inputs = processor(image, input_boxes=box, return_tensors="pt").to("cuda")

            with torch.no_grad():
                outputs = model(**inputs)
                masks = processor.image_processor.post_process_masks(
                    outputs.pred_masks.cpu(),
                    inputs["original_sizes"].cpu(),
                    inputs["reshaped_input_sizes"].cpu()
                )
                mask_tensor = masks[0].squeeze(0)
                mask = mask_tensor[0].numpy().astype(np.uint8)

            masked_obj = np.array(image) * mask[:, :, None]
            alpha = (mask * 255).astype(np.uint8)
            obj_rgba = np.dstack((masked_obj, alpha))

            x1, y1, x2, y2 = item["box"]
            obj_crop = obj_rgba[y1:y2, x1:x2]
            obj_pil = Image.fromarray(obj_crop)

            # === Scale vị trí và kích thước ===
            scaled_x1 = int(x1 * scale_x)
            scaled_y1 = int(y1 * scale_y)
            scaled_x2 = int(x2 * scale_x)
            scaled_y2 = int(y2 * scale_y)

            new_w = scaled_x2 - scaled_x1
            new_h = scaled_y2 - scaled_y1
            obj_pil = obj_pil.resize((new_w, new_h), Image.LANCZOS)

            bg_copy.paste(obj_pil, (scaled_x1, scaled_y1), obj_pil)
            annotations.append((item["label"], [scaled_x1, scaled_y1, scaled_x2, scaled_y2]))

        # === Lưu ảnh và XML ===
        base_name = os.path.splitext(image_file)[0]
        bg_name = os.path.splitext(bg_file)[0]
        out_image_name = f"{base_name}_on_{bg_name}.png"
        out_image_path = os.path.join(output_dir, out_image_name)
        bg_copy.save(out_image_path)

        xml_out_path = os.path.join(output_dir, f"{base_name}_on_{bg_name}.xml")
        create_voc_xml(
            filename=out_image_name,
            path=out_image_path,
            width=bg_copy.width,
            height=bg_copy.height,
            depth=3,
            labels_boxes=annotations,
            save_path=xml_out_path
        )

        print(f"✅ Đã lưu: {out_image_path}")
        print(f"📝 Annotation: {xml_out_path}")
